# Cross-Site Classifier Analysis

**Train on DRIAMS-A (University Hospital Basel), test on B (Canton Basel-Land), C (Canton Aarau), D (Viollier).**

Cross-site generalisation: can a model trained on one hospital predict AMR at completely different hospitals?

**Approaches:** LR (L2 + PCA+L2, threshold tuned) and MLP (Regularised + Attention, threshold tuned).
**Preprocessing:** log1p + standardise (fit on A train only).
**Split:** Species-stratified 80/20 on A; B/C/D used as-is for testing.

In [ ]:
!pip install maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldideepkit.attention.mlp import MaldiMLPClassifier
from maldiamrkit.evaluation import stratified_species_drug_split

from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                              ConfusionMatrixDisplay, RocCurveDisplay)

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")

OUT_DIR = Path("./results_cross_site")
OUT_DIR.mkdir(exist_ok=True)

TRAIN_SITE = "Proc_DRIAMS-A"
TEST_SITES = ["Proc_DRIAMS-B", "Proc_DRIAMS-C", "Proc_DRIAMS-D"]
BIN_COLS = [f"bin_{i}" for i in range(6000)]

DRUGS = [
    "Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic_acid",
    "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
    "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin",
]

print(f"Train: {TRAIN_SITE}  |  Test: {TEST_SITES}  |  Drugs: {len(DRUGS)}")

In [ ]:
# =============================================================================
# 1. LOAD DATA: separate A from B/C/D
# =============================================================================

def load_site_drug(site, drug):
    p = DATA_ROOT / site / drug / "data.csv"
    if not p.exists():
        return None, None, None
    df = pd.read_csv(p)
    X = df[BIN_COLS].values.astype("float32")
    y = df["label"].values.astype(int)
    species = df["species"].values
    return X, y, species


def create_train_split(X, y, species, val_size=0.20, seed=SEED):
    n = len(y)
    idx = np.arange(n)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx.reshape(-1, 1), y, species=species, test_size=val_size, random_state=seed)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    return {"train": (X[idx_train], y[idx_train]),
            "val":   (X[idx_val],   y[idx_val])}


print("Loading...")
drug_data = {}

for drug in tqdm(DRUGS, desc="Loading"):
    X_a, y_a, sp_a = load_site_drug(TRAIN_SITE, drug)
    if X_a is None:
        print(f"  {drug}: no data in A, skipping")
        continue
    a_split = create_train_split(X_a, y_a, sp_a)

    test_sets = {}
    for site in TEST_SITES:
        X_s, y_s, _ = load_site_drug(site, drug)
        if X_s is not None:
            test_sets[site] = (X_s, y_s)

    drug_data[drug] = {"a_split": a_split, "test_sets": test_sets}
    total_test = sum(t[0].shape[0] for t in test_sets.values())
    print(f"  {drug:35s}  A: train={a_split['train'][0].shape[0]} val={a_split['val'][0].shape[0]}  "
          f"test sites={list(test_sets.keys())} ({total_test} samples)")
print("Done.")

---
## Logistic Regression Cross-Site

In [ ]:
# =============================================================================
# 2. CROSS-SITE PIPELINE: Train on A, test on B/C/D
# =============================================================================

thresholds = np.linspace(0.05, 0.95, 91)
C_grid = np.linspace(5e-5, 1e-3, 15)

lr_results = {}

for drug in tqdm(DRUGS, desc="LR"):
    if drug not in drug_data: continue
    d = drug_data[drug]
    a_split = d["a_split"]
    test_sets = d["test_sets"]
    X_train_a, y_train_a = a_split["train"]
    X_val_a, y_val_a = a_split["val"]

    state = fit_input_transform(X_train_a, "log1p+standardize")
    pp = lambda X: apply_input_transform(X, state)
    X_train_pp = pp(X_train_a); X_val_pp = pp(X_val_a)

    test_pp = {}
    for site, (X_s, y_s) in test_sets.items():
        test_pp[site] = (pp(X_s), y_s)
    if len(test_pp) > 0:
        X_all = np.concatenate([t[0] for t in test_pp.values()])
        y_all = np.concatenate([t[1] for t in test_pp.values()])
        test_pp["B+C+D"] = (X_all, y_all)

    lr_results[drug] = {}

    # L2 LR + tuned
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid.fit(X_train_pp, y_train_a)
    lr_l2 = grid.best_estimator_
    proba_val_l2 = lr_l2.predict_proba(X_val_pp)[:, 1]
    best_t_l2 = thresholds[np.argmax([balanced_accuracy_score(y_val_a, proba_val_l2 >= t) for t in thresholds])]

    app = "L2 LR (tuned)"
    lr_results[drug][app] = {}
    for st, (X_st, y_st) in [("A-val", (X_val_pp, y_val_a))] + list(test_pp.items()):
        preds = (lr_l2.predict_proba(X_st)[:, 1] >= best_t_l2)
        lr_results[drug][app][st] = {"BalAcc": balanced_accuracy_score(y_st, preds),
                                      "AUC": roc_auc_score(y_st, lr_l2.predict_proba(X_st)[:, 1])}

    # PCA+L2 LR + tuned
    scaler = StandardScaler().fit(X_train_pp)
    pca = PCA(n_components=0.94, random_state=SEED).fit(scaler.transform(X_train_pp))
    X_train_pca = pca.transform(scaler.transform(X_train_pp))
    X_val_pca = pca.transform(scaler.transform(X_val_pp))

    grid_pca = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid_pca.fit(X_train_pca, y_train_a)
    lr_pca = grid_pca.best_estimator_
    proba_val_pca = lr_pca.predict_proba(X_val_pca)[:, 1]
    best_t_pca = thresholds[np.argmax([balanced_accuracy_score(y_val_a, proba_val_pca >= t) for t in thresholds])]

    app2 = "PCA+L2 LR (tuned)"
    lr_results[drug][app2] = {}
    for st, (X_st, y_st) in [("A-val", (X_val_pp, y_val_a))] + list(test_pp.items()):
        X_pca_st = X_val_pca if st == "A-val" else pca.transform(scaler.transform(X_st))
        preds = (lr_pca.predict_proba(X_pca_st)[:, 1] >= best_t_pca)
        lr_results[drug][app2][st] = {"BalAcc": balanced_accuracy_score(y_st, preds),
                                       "AUC": roc_auc_score(y_st, lr_pca.predict_proba(X_pca_st)[:, 1])}

print("\nLR cross-site done.")

In [ ]:
# =============================================================================
# 3. LR CROSS-SITE HEATMAP
# =============================================================================

sites_order = ["A-val", "Proc_DRIAMS-B", "Proc_DRIAMS-C", "Proc_DRIAMS-D", "B+C+D"]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax_idx, method in enumerate(["L2 LR (tuned)", "PCA+L2 LR (tuned)"]):
    data = {}
    for drug in DRUGS:
        if drug not in lr_results: continue
        row = []
        for st in sites_order:
            row.append(lr_results[drug][method].get(st, {}).get("BalAcc", np.nan))
        data[drug] = row
    hm = pd.DataFrame(data, index=sites_order).T
    sns.heatmap(hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
                linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=axes[ax_idx])
    axes[ax_idx].set_title(f"LR {method} -- Train A -> Test B/C/D")
    axes[ax_idx].set_ylabel("Drug"); axes[ax_idx].set_xlabel("Test Set")

plt.tight_layout()
plt.savefig(OUT_DIR / "lr_cross_site_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nLR Cross-Site Summary:")
rows = []
for drug in DRUGS:
    if drug not in lr_results: continue
    for m in ["L2 LR (tuned)", "PCA+L2 LR (tuned)"]:
        r = {"Drug": drug, "Method": m.split()[0]}
        for st in sites_order:
            r[st] = lr_results[drug][m].get(st, {}).get("BalAcc", np.nan)
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

---
## MLP Cross-Site

In [ ]:
# =============================================================================
# 4. MLP CROSS-SITE
# =============================================================================

LR_GRID   = np.linspace(7.5e-5, 9.5e-5, 8)
DROP_GRID = np.linspace(0.5, 0.8, 8)
mlp_results = {}

for drug in tqdm(DRUGS, desc="MLP"):
    if drug not in drug_data: continue
    d = drug_data[drug]
    a_split = d["a_split"]
    test_sets = d["test_sets"]
    X_train_a, y_train_a = a_split["train"]
    X_val_a, y_val_a = a_split["val"]

    state = fit_input_transform(X_train_a, "log1p+standardize")
    pp = lambda X: apply_input_transform(X, state)
    X_train_pp = pp(X_train_a); X_val_pp = pp(X_val_a)

    test_pp = {}
    for site, (X_s, y_s) in test_sets.items():
        test_pp[site] = (pp(X_s), y_s)
    if len(test_pp) > 0:
        X_all = np.concatenate([t[0] for t in test_pp.values()])
        y_all = np.concatenate([t[1] for t in test_pp.values()])
        test_pp["B+C+D"] = (X_all, y_all)

    # Grid search
    best_balacc = -1; best_lr = None; best_dh = None; best_dl = None
    for lr in LR_GRID:
        for d in DROP_GRID:
            dh, dl = d, d / 2
            mlp_tmp = MaldiMLPClassifier(
                hidden_dim=512, head_dims=(256, 128), use_attention=False,
                dropout_high=dh, dropout_low=dl, weight_decay=1e-3,
                learning_rate=lr, batch_size=64, epochs=50,
                early_stopping_patience=10, warmup_epochs=10,
                val_fraction=0.1, use_sam=False,
                input_transform="none", tune_threshold=False,
                random_state=SEED, verbose=False)
            mlp_tmp.fit(X_train_pp, y_train_a)
            val_ba = balanced_accuracy_score(y_val_a, mlp_tmp.predict(X_val_pp))
            if val_ba > best_balacc:
                best_balacc = val_ba; best_lr = lr; best_dh = dh; best_dl = dl

    print(f"  {drug:35s} best lr={best_lr:.1e} drop=({best_dh:.2f},{best_dl:.2f}) val_balacc={best_balacc:.4f}")
    mlp_results[drug] = {}

    # Regularised MLP
    mlp_reg = MaldiMLPClassifier(
        hidden_dim=512, head_dims=(256, 128), use_attention=False,
        dropout_high=best_dh, dropout_low=best_dl, weight_decay=1e-4,
        learning_rate=best_lr, batch_size=64, epochs=50,
        early_stopping_patience=10, warmup_epochs=10,
        val_fraction=0.1, use_sam=False,
        input_transform="none", tune_threshold=False, random_state=SEED, verbose=False)
    mlp_reg.fit(X_train_pp, y_train_a)

    pv_r = mlp_reg.predict_proba(X_val_pp)[:, 1]
    bt_r = thresholds[np.argmax([balanced_accuracy_score(y_val_a, pv_r >= t) for t in thresholds])]
    app = "Reg MLP (tuned)"
    mlp_results[drug][app] = {}
    for st, (X_st, y_st) in [("A-val", (X_val_pp, y_val_a))] + list(test_pp.items()):
        preds = (mlp_reg.predict_proba(X_st)[:, 1] >= bt_r)
        mlp_results[drug][app][st] = {"BalAcc": balanced_accuracy_score(y_st, preds),
                                       "AUC": roc_auc_score(y_st, mlp_reg.predict_proba(X_st)[:, 1])}

    # Attention MLP
    mlp_attn = MaldiMLPClassifier(
        hidden_dim=512, head_dims=(256, 128), use_attention=True,
        dropout_high=0.4, dropout_low=0.2, weight_decay=1e-3,
        learning_rate=1e-3, batch_size=32, epochs=100,
        early_stopping_patience=10, val_fraction=0.1,
        input_transform="none", tune_threshold=False, random_state=SEED, verbose=False)
    mlp_attn.fit(X_train_pp, y_train_a)

    pv_a = mlp_attn.predict_proba(X_val_pp)[:, 1]
    bt_a = thresholds[np.argmax([balanced_accuracy_score(y_val_a, pv_a >= t) for t in thresholds])]
    app2 = "Attn MLP (tuned)"
    mlp_results[drug][app2] = {}
    for st, (X_st, y_st) in [("A-val", (X_val_pp, y_val_a))] + list(test_pp.items()):
        preds = (mlp_attn.predict_proba(X_st)[:, 1] >= bt_a)
        mlp_results[drug][app2][st] = {"BalAcc": balanced_accuracy_score(y_st, preds),
                                        "AUC": roc_auc_score(y_st, mlp_attn.predict_proba(X_st)[:, 1])}

print("\nMLP cross-site done.")

In [ ]:
# =============================================================================
# 5. MLP CROSS-SITE HEATMAP
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax_idx, method in enumerate(["Reg MLP (tuned)", "Attn MLP (tuned)"]):
    data = {}
    for drug in DRUGS:
        if drug not in mlp_results: continue
        row = [mlp_results[drug][method].get(st, {}).get("BalAcc", np.nan) for st in sites_order]
        data[drug] = row
    hm = pd.DataFrame(data, index=sites_order).T
    sns.heatmap(hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
                linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=axes[ax_idx])
    axes[ax_idx].set_title(f"MLP {method} -- Train A -> Test B/C/D")
    axes[ax_idx].set_ylabel("Drug"); axes[ax_idx].set_xlabel("Test Set")

plt.tight_layout()
plt.savefig(OUT_DIR / "mlp_cross_site_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nMLP Cross-Site Summary:")
rows = []
for drug in DRUGS:
    if drug not in mlp_results: continue
    for m in ["Reg MLP (tuned)", "Attn MLP (tuned)"]:
        r = {"Drug": drug, "Method": m}
        for st in sites_order:
            r[st] = mlp_results[drug][m].get(st, {}).get("BalAcc", np.nan)
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

---
## LR vs MLP -- Best per Drug (B+C+D combined)

In [ ]:
# =============================================================================
# 6. LR vs MLP -- BEST PER DRUG (tested on B+C+D)
# =============================================================================

summary_rows = []
for drug in DRUGS:
    if drug not in lr_results or drug not in mlp_results: continue
    for method, results in [("LR", lr_results), ("MLP", mlp_results)]:
        for approach in results[drug]:
            if "B+C+D" in results[drug][approach]:
                summary_rows.append({
                    "Drug": drug, "Method": method, "Approach": approach,
                    "Test_BalAcc": results[drug][approach]["B+C+D"]["BalAcc"],
                    "Test_AUC": results[drug][approach]["B+C+D"]["AUC"],
                    "A-val_BalAcc": results[drug][approach].get("A-val", {}).get("BalAcc", np.nan),
                })

df_sum = pd.DataFrame(summary_rows)
best = df_sum.loc[df_sum.groupby("Drug")["Test_BalAcc"].idxmax()]
print("\nBest approach per drug (B+C+D combined):")
for _, r in best.iterrows():
    print(f"  {r['Drug']:35s} {r['Method']:3s} {r['Approach']:20s}  "
          f"Test BalAcc={r['Test_BalAcc']:.4f}  AUC={r['Test_AUC']:.4f}")

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(best)); w = 0.3
ax.bar(x - w/2, best["Test_BalAcc"], w, label="Test BalAcc (B+C+D)", color="#1f77b4")
ax.bar(x + w/2, best["Test_AUC"], w, label="Test AUC (B+C+D)", color="#ff7f0e")
ax.scatter(x, best["A-val_BalAcc"], marker="v", color="red", s=60, label="A-val BalAcc")
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Drug'][:15]}\n{r['Method']} {r['Approach'][:12]}" for _, r in best.iterrows()], fontsize=7)
ax.set_ylabel("Score"); ax.set_title("Cross-Site Generalisation -- Best Approach per Drug (A->B+C+D)")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "cross_site_best_per_drug.pdf")
plt.show()

In [ ]:
print("\nDone. Cross-site results saved to", OUT_DIR.resolve())
print(f"Drugs analysed: {len([d for d in DRUGS if d in lr_results])}")
for f in sorted(OUT_DIR.glob("*.pdf")):
    print(f"  {f.name}")